In [ ]:
import shutil, os

if os.path.exists("output"): 
    shutil.rmtree("output")
print("output directory cleared")

In [ ]:
!pip install transformers seqeval sentencepiece protobuf evaluate safetensors bitsandbytes -q

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token)

In [ ]:
import os, shutil

os.makedirs("data/bio_dataset_deberta", exist_ok=True)
base_path = "/kaggle/input/datasets/kaushikkumarceg/gsoc26-proposal-finetuning-deberta-dataset"

for file_name in ["train.jsonl", "val.jsonl", "test.jsonl", "label_map.json"]:
    shutil.copy(f"{base_path}/{file_name}", f"data/bio_dataset_deberta/{file_name}")

print("dataset loaded")

In [ ]:
#DeBERTa-v3-large BIO finetuning for scancode phrase extraction - gsoc proposal'26 (implements LLRD and handles heavy class imbalance)

import json
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
import evaluate

# Handle 8-bit optimization if bitsandbytes is available
try:
    from bitsandbytes.optim import AdamW8bit
    USE_8BIT = True
except ImportError:
    from torch.optim import AdamW as AdamW8bit
    USE_8BIT = False

# Configuration
MODEL_NAME    = "microsoft/deberta-v3-large"
LABEL2ID      = {"O": 0, "B-REQ": 1, "I-REQ": 2}
ID2LABEL      = {0: "O", 1: "B-REQ", 2: "I-REQ"}
IGNORE        = -100
NUM_LABELS    = 3

# Training Hyperparameters
CLASS_WEIGHTS    = [1.0, 2.0, 1.5]
ENTITY_MASK_PROB = 0.0 
DATA_DIR         = Path("data/bio_dataset_deberta")
OUTPUT_DIR       = Path("output/deberta_large_run7")

EPOCHS       = 8
BATCH_SIZE   = 1
GRAD_ACCUM   = 16 
BASE_LR      = 2e-5
DECAY_RATE   = 0.98
WARMUP_RATIO = 0.1
NUM_LAYERS   = 24

# Custom Loss and Trainer
class StableCELoss(nn.Module):
    """
    Standard CrossEntropy using LogSumExp for numerical stability.
    Weighted to handle class imbalance in license rules.
    """
    def __init__(self, weight):
        super().__init__()
        self.weight = weight

    def forward(self, logits, labels):
        weight = self.weight.to(logits.device).to(logits.dtype)
        return nn.functional.cross_entropy(
            logits, labels, weight=weight, ignore_index=IGNORE
        )

class StableCETrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # DeBERTa-v3 does not use token_type_ids; remove to prevent embedding corruption
        inputs.pop("token_type_ids", None)
        
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        loss    = StableCELoss(self.class_weights)(
            logits.view(-1, NUM_LABELS), labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

# Optimization Strategy
def get_llrd_optimizer(model):
    """
    Implements Layer-wise Learning Rate Decay (LLRD).
    Higher layers (near the head) get a higher LR; lower layers get decayed LR.
    """
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    groups   = []

    # 1. Classifier Head
    for nd_flag in [False, True]:
        params = [p for n, p in model.named_parameters()
                  if "classifier" in n and (any(nd in n for nd in no_decay) == nd_flag)
                  and p.requires_grad]
        if params:
            groups.append({
                "params": params, 
                "weight_decay": 0.0 if nd_flag else 0.01, 
                "lr": BASE_LR
            })

    # 2.Transformer Layers (23 down to 0)
    for layer in range(NUM_LAYERS - 1, -1, -1):
        lr = BASE_LR * (DECAY_RATE ** (NUM_LAYERS - layer))
        for nd_flag in [False, True]:
            params = [p for n, p in model.named_parameters()
                      if f"encoder.layer.{layer}." in n
                      and (any(nd in n for nd in no_decay) == nd_flag)
                      and p.requires_grad]
            if params:
                groups.append({
                    "params": params, 
                    "weight_decay": 0.0 if nd_flag else 0.01, 
                    "lr": lr
                })

    # 3.Embeddings
    embed_lr = BASE_LR * (DECAY_RATE ** (NUM_LAYERS + 1))
    for nd_flag in [False, True]:
        params = [p for n, p in model.named_parameters()
                  if ("embeddings" in n or "rel_embeddings" in n)
                  and "encoder.layer" not in n
                  and (any(nd in n for nd in no_decay) == nd_flag)
                  and p.requires_grad]
        if params:
            groups.append({
                "params": params, 
                "weight_decay": 0.0 if nd_flag else 0.01, 
                "lr": embed_lr
            })

    opt_cls = AdamW8bit if USE_8BIT else torch.optim.AdamW
    return opt_cls(groups, lr=BASE_LR, eps=1e-6, betas=(0.9, 0.999))

# Data Handling
class BIODataset(Dataset):
    def __init__(self, jsonl_path, mask_token_id, train=False):
        self.examples = []
        self.train   = train
        self.mask_id = mask_token_id
        with open(jsonl_path) as f:
            for line in f:
                ex = json.loads(line)["deberta_large"]
                self.examples.append({
                    "input_ids": ex["input_ids"],
                    "attention_mask": ex["attention_mask"],
                    "labels": ex["labels"],
                })

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        item = {k: list(v) for k, v in self.examples[idx].items()}
        # Optional: Randomly mask entities during training to force context learning
        if self.train and ENTITY_MASK_PROB > 0:
            labels = item["labels"]
            for i in range(len(labels)):
                if labels[i] != 0 and random.random() < ENTITY_MASK_PROB:
                    item["input_ids"][i] = self.mask_id
                    
        return {k: torch.tensor(v) for k, v in item.items()}

# Prediction and Metrics
def constrained_decode(logits_seq, active_mask):
    """
    Prevents invalid BIO transitions (e.g., O -> I-REQ).
    """
    preds = np.argmax(logits_seq, axis=1)
    tags, prev = [], "O"
    for i, (pred, active) in enumerate(zip(preds, active_mask)):
        if not active: continue
        tag = ID2LABEL[pred]
        if tag == "I-REQ" and prev == "O":
            # if I follows O, treat it as a B or keep O
            tag = "B-REQ" 
        tags.append(tag)
        prev = tag
    return tags

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    tp, tl = [], []
    for i in range(len(labels)):
        active = labels[i] != IGNORE
        tp.append(constrained_decode(logits[i], active))
        tl.append([ID2LABEL[l] for l in labels[i] if l != IGNORE])
    
    results = seqeval.compute(predictions=tp, references=tl)
    return {
        "precision": round(results["overall_precision"], 4),
        "recall":    round(results["overall_recall"], 4),
        "f1":        round(results["overall_f1"], 4),
        "accuracy":  round(results["overall_accuracy"], 4)
    }

# Training Loop
def average_checkpoints(output_dir, num_ckpts=2):
    """
    Simple stochastic weight averaging of the last N checkpoints.
    """
    import glob
    from safetensors.torch import load_file as load_st

    ckpt_paths = sorted(glob.glob(str(output_dir / "checkpoint-*")),
                        key=lambda x: int(x.split("-")[-1]))[-num_ckpts:]
    
    if len(ckpt_paths) < 2: return None

    states = []
    for p in ckpt_paths:
        path = Path(p) / "model.safetensors"
        if path.exists():
            states.append(load_st(str(path), device="cpu"))
    
    if len(states) < 2: return None
    return {k: torch.stack([s[k].float() for s in states]).mean(0) for k in states[0]}

def run_training(hf_token=None):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=NUM_LABELS,
        id2label=ID2LABEL, 
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True
    )
    model.gradient_checkpointing_enable()

    train_ds = BIODataset(DATA_DIR/"train.jsonl", tokenizer.mask_token_id, train=True)
    val_ds   = BIODataset(DATA_DIR/"val.jsonl", tokenizer.mask_token_id)
    test_ds  = BIODataset(DATA_DIR/"test.jsonl", tokenizer.mask_token_id)

    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=BASE_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=0.5,
        gradient_checkpointing=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=2,
        dataloader_num_workers=2,
        report_to="none"
    )

    trainer = StableCETrainer(
        class_weights=CLASS_WEIGHTS,
        model=model, 
        args=args,
        train_dataset=train_ds, 
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer, label_pad_token_id=IGNORE),
        compute_metrics=compute_metrics,
        optimizers=(get_llrd_optimizer(model), None),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
    )

    trainer.train()

    #Average top weights and evaluate on test set
    avg_weights = average_checkpoints(OUTPUT_DIR)
    if avg_weights:
        model.load_state_dict(avg_weights, strict=False)

    test_results = trainer.evaluate(test_ds)
    print(f"\nTest Results: {test_results}")

    if hf_token:
        repo = "scancode-required-phrases-deberta-large"
        model.push_to_hub(repo, token=hf_token)
        tokenizer.push_to_hub(repo, token=hf_token)

    return test_results

if __name__ == "__main__":
    run_training()

In [ ]:
train(hf_token=token)